In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. Fetch monthly USD/TRY data
data_monthly = yf.download("TRY=X", start="2010-01-01", interval="1mo")

# Select only the Close price and make a clean copy (fixes SettingWithCopyWarning)
df_m = data_monthly[['Close']].copy()
df_m.columns = ['USDTRY']
df_m.dropna(inplace=True)

# Set monthly frequency on the index (fixes ValueWarning)
df_m.index = pd.DatetimeIndex(df_m.index).to_period('M').to_timestamp()
df_m = df_m.asfreq('MS')

print(df_m.head())
print(f"\nMissing values: {df_m.isnull().sum().values[0]}")

# Visualize
df_m.plot(figsize=(10, 5), title="USD/TRY Exchange Rate (2010-2026)")
plt.ylabel("Exchange Rate (TL)")
plt.tight_layout()
plt.show()

In [ ]:
# 2. Stationarity Test (ADF) on raw series
result = adfuller(df_m['USDTRY'])

print("--- ADF Test (Raw Series) ---")
print(f"ADF Statistic : {result[0]:.4f}")
print(f"p-value       : {result[1]:.4f}")
print("Critical Values:")
for key, value in result[4].items():
    print(f"  {key}: {value:.4f}")
print("\nConclusion: Series is NON-stationary (p > 0.05). Transformation required.")

In [ ]:
# 3. Logarithmic transformation to achieve stationarity (fixes SettingWithCopyWarning)
df_log = np.log(df_m['USDTRY'] / df_m['USDTRY'].shift(1)).dropna()

# ADF Test on transformed series
result_diff = adfuller(df_log)

print("--- ADF Test (Log Returns) ---")
print(f"ADF Statistic : {result_diff[0]:.4f}")
print(f"p-value       : {result_diff[1]:.4f}")
print("\nConclusion: Series is STATIONARY (p < 0.05). Ready for ARIMA.")

# Visualize stationary series
df_log.plot(figsize=(10, 4), title="USD/TRY Logarithmic Returns")
plt.ylabel("Log Return")
plt.tight_layout()
plt.show()

In [ ]:
# 4. ACF and PACF plots (to determine p and q parameters)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

plot_acf(df_log, lags=40, ax=ax1)
ax1.set_title('Autocorrelation Function (ACF) — determines q (MA term)')

plot_pacf(df_log, lags=40, ax=ax2)
ax2.set_title('Partial Autocorrelation Function (PACF) — determines p (AR term)')

plt.tight_layout()
plt.show()

In [ ]:
# 5. Model selection via AIC
model1 = ARIMA(df_log, order=(1, 0, 1)).fit()
model2 = ARIMA(df_log, order=(2, 0, 2)).fit()

print(f"ARIMA(1,0,1) AIC: {model1.aic:.4f}")
print(f"ARIMA(2,0,2) AIC: {model2.aic:.4f}")
print(f"\nSelected model: ARIMA(1,0,1) — lower AIC is better")

In [ ]:
# 6. Final model summary
model_fit = ARIMA(df_log, order=(1, 0, 1)).fit()
print(model_fit.summary())

In [ ]:
# 7. Residual diagnostics
model_fit.plot_diagnostics(figsize=(12, 8))
plt.suptitle('ARIMA(1,0,1) Residual Diagnostics', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 8. 3-Month Forecast
forecast_steps = 3
final_forecast_obj = model_fit.get_forecast(steps=forecast_steps)
final_forecast_df = final_forecast_obj.summary_frame()

# Revert log-returns back to actual price scale
last_price = df_m['USDTRY'].iloc[-1]
final_prices = []
conf_lower = []
conf_upper = []
cumulative_log_val = 0

for i, row in final_forecast_df.iterrows():
    cumulative_log_val += row['mean']
    final_prices.append(last_price * np.exp(cumulative_log_val))
    conf_lower.append(last_price * np.exp(cumulative_log_val - 1.96 * row['mean_se']))
    conf_upper.append(last_price * np.exp(cumulative_log_val + 1.96 * row['mean_se']))

forecast_dates = pd.date_range(start=df_m.index[-1], periods=forecast_steps + 1, freq='MS')[1:]
final_prediction_table = pd.DataFrame({
    'Forecast USD/TRY': final_prices,
    'Lower 95% CI': conf_lower,
    'Upper 95% CI': conf_upper
}, index=forecast_dates)

print("--- 3-MONTH USD/TRY EXCHANGE RATE FORECAST ---")
print(final_prediction_table.round(4))

In [ ]:
# 9. Final visualization with confidence intervals
actual_data_tail = df_m['USDTRY'].tail(12)

plt.figure(figsize=(12, 6))

plt.plot(actual_data_tail.index, actual_data_tail.values,
         label='Historical Data (Last 12 Months)', color='blue', marker='o')

plt.plot(final_prediction_table.index, final_prediction_table['Forecast USD/TRY'],
         label='3-Month Forecast', color='red', linestyle='--', marker='s')

plt.fill_between(final_prediction_table.index,
                 final_prediction_table['Lower 95% CI'],
                 final_prediction_table['Upper 95% CI'],
                 color='red', alpha=0.15, label='95% Confidence Interval')

plt.axvspan(actual_data_tail.index[-1], final_prediction_table.index[-1],
            color='gray', alpha=0.1)

plt.title('USD/TRY Exchange Rate: Historical vs Forecasted', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Exchange Rate (TL)', fontsize=12)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()